##Przygotowanie zbioru i modelu

In [ ]:
import pandas as pd

df = pd.read_csv('fake reviews dataset.csv')

df['label'] = df['label'].map({'CG': 0, 'OR': 1})

!pip install transformers==4.53.0

from transformers import BertTokenizer, TFBertForSequenceClassification
import tensorflow as tf
from sklearn.model_selection import train_test_split
import pandas as pd

from sklearn.model_selection import train_test_split

X = df['text_']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def encode_text(texts):
    return tokenizer(texts.tolist(), padding=True, truncation=True, max_length=256, return_tensors='tf')

X_train_enc = encode_text(X_train)
X_test_enc = encode_text(X_test)

model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Freeze glownej warstwy BERT zeby uczyly sie wszystkie oprócz TFBertMainLayer
for layer in model.layers[:-2]:
    layer.trainable = False

# stworzenie warstwy wejscia i maski informujacej o paddingu
input_ids = tf.keras.Input(shape=(256,), dtype=tf.int32, name='input_ids')
attention_mask = tf.keras.Input(shape=(256,), dtype=tf.int32, name='attention_mask')

bert_output = model.bert(input_ids, attention_mask=attention_mask)[1]
# [1] to pooled output czyli wektor reprezentujacy cala sekwencje
dropout = tf.keras.layers.Dropout(0.2)(bert_output)
#dropout to losowe wylaczenie neuronow (tu 30%), zmniejsza ryzyko przeuczenia modelu
dense_1 = tf.keras.layers.Dense(128, activation='relu')(dropout)
dropout_2 = tf.keras.layers.Dropout(0.3)(dense_1)
output = tf.keras.layers.Dense(1, activation='sigmoid')(dropout_2)

final_model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)
final_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
                    loss='binary_crossentropy',
                    metrics=['accuracy'])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 101.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('fake reviews dataset.csv')

In [ ]:
df.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [ ]:
counts = df['label'].value_counts()
print("Liczba recenzji CG (falszywe):", counts.get('CG', 0))
print("Liczba recenzji OR (prawdziwe):", counts.get('OR', 0))

Liczba recenzji CG (falszywe): 20216
Liczba recenzji OR (prawdziwe): 20216


In [ ]:
# Wczytanie danych
df = pd.read_csv('fake reviews dataset.csv')
# Liczenie duplikatów
ilosc_duplikatow = df.duplicated(subset='text_').sum()

print(ilosc_duplikatow)

20


Zamiana CG na 0, i OR na 1

In [ ]:
df['label'] = df['label'].map({'CG': 0, 'OR': 1})

In [ ]:
# ilosc wartosci 0 i 1
print(df['label'].value_counts())

label
0    20216
1    20216
Name: count, dtype: int64


In [ ]:
# podzial na set treningowy i testowy
from sklearn.model_selection import train_test_split

X = df['text_']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

importy do modelu

In [ ]:
!pip install transformers==4.53.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 120.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 124.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=

In [ ]:
from transformers import BertTokenizer, TFBertForSequenceClassification
import tensorflow as tf
from sklearn.model_selection import train_test_split
import pandas as pd

tokenizacja

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def encode_text(texts):
    return tokenizer(texts.tolist(), padding=True, truncation=True, max_length=256, return_tensors='tf')

X_train_enc = encode_text(X_train)
X_test_enc = encode_text(X_test)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


budowa i trening modelu

In [ ]:
model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Freeze glownej warstwy BERT zeby uczyly sie wszystkie oprócz TFBertMainLayer
for layer in model.layers[:-2]:
    layer.trainable = False

# stworzenie warstwy wejscia i maski informujacej o paddingu
input_ids = tf.keras.Input(shape=(256,), dtype=tf.int32, name='input_ids')
attention_mask = tf.keras.Input(shape=(256,), dtype=tf.int32, name='attention_mask')

bert_output = model.bert(input_ids, attention_mask=attention_mask)[1]
# [1] to pooled output czyli wektor reprezentujacy cala sekwencje
dropout = tf.keras.layers.Dropout(0.2)(bert_output)
#dropout to losowe wylaczenie neuronow (tu 30%), zmniejsza ryzyko przeuczenia modelu
dense_1 = tf.keras.layers.Dense(128, activation='relu')(dropout)
dropout_2 = tf.keras.layers.Dropout(0.3)(dense_1)
output = tf.keras.layers.Dense(1, activation='sigmoid')(dropout_2)

final_model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)
final_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
                    loss='binary_crossentropy',
                    metrics=['accuracy'])

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# wiekszy trening test

In [ ]:
history = final_model.fit(
  [X_train_enc['input_ids'], X_train_enc['attention_mask']],
  y_train,
  epochs=6,
  batch_size=10,
  validation_data=([X_test_enc['input_ids'], X_test_enc['attention_mask']], y_test)
)

Epoch 1/6
3235/3235 [==============================] - 1024s 307ms/step - loss: 0.6602 - accuracy: 0.6023 - val_loss: 0.5628 - val_accuracy: 0.7568
Epoch 2/6
3235/3235 [==============================] - 998s 308ms/step - loss: 0.5844 - accuracy: 0.6887 - val_loss: 0.5139 - val_accuracy: 0.7526
Epoch 3/6
3235/3235 [==============================] - 1008s 311ms/step - loss: 0.5419 - accuracy: 0.7250 - val_loss: 0.5318 - val_accuracy: 0.7089
Epoch 4/6
3235/3235 [==============================] - 994s 307ms/step - loss: 0.5139 - accuracy: 0.7431 - val_loss: 0.4527 - val_accuracy: 0.7879
Epoch 5/6
3235/3235 [==============================] - 999s 309ms/step - loss: 0.4998 - accuracy: 0.7538 - val_loss: 0.4545 - val_accuracy: 0.7780
Epoch 6/6
3235/3235 [==============================] - 1009s 312ms/step - loss: 0.4867 - accuracy: 0.7615 - val_loss: 0.4411 - val_accuracy: 0.7866
Epoch 1/6
2696/2696 [==============================] - 1006s 373ms/step - loss: 0.4778 - accuracy: 0.7681 - val_los

In [ ]:
history = final_model.fit(
  [X_train_enc['input_ids'], X_train_enc['attention_mask']],
  y_train,
  epochs=6,
  batch_size=12,
  validation_data=([X_test_enc['input_ids'], X_test_enc['attention_mask']], y_test)
)

Epoch 1/6
2696/2696 [==============================] - 900s 322ms/step - loss: 0.6698 - accuracy: 0.5957 - val_loss: 0.5818 - val_accuracy: 0.6930
Epoch 2/6
2696/2696 [==============================] - 867s 322ms/step - loss: 0.5869 - accuracy: 0.6875 - val_loss: 0.5201 - val_accuracy: 0.7528
Epoch 3/6
2696/2696 [==============================] - 867s 321ms/step - loss: 0.5499 - accuracy: 0.7185 - val_loss: 0.4920 - val_accuracy: 0.7612
Epoch 4/6
2696/2696 [==============================] - 868s 322ms/step - loss: 0.5252 - accuracy: 0.7357 - val_loss: 0.4937 - val_accuracy: 0.7471
Epoch 5/6
2696/2696 [==============================] - 864s 321ms/step - loss: 0.5055 - accuracy: 0.7499 - val_loss: 0.4957 - val_accuracy: 0.7435
Epoch 6/6
2696/2696 [==============================] - 866s 321ms/step - loss: 0.4917 - accuracy: 0.7582 - val_loss: 0.4755 - val_accuracy: 0.7604
Epoch 1/6
2311/2311 [==============================] - 893s 386ms/step - loss: 0.4813 - accuracy: 0.7648 - val_loss: 0

In [ ]:
history = final_model.fit(
  [X_train_enc['input_ids'], X_train_enc['attention_mask']],
  y_train,
  epochs=6,
  batch_size=14,
  validation_data=([X_test_enc['input_ids'], X_test_enc['attention_mask']], y_test)
)

Epoch 1/6
2311/2311 [==============================] - 945s 402ms/step - loss: 0.6735 - accuracy: 0.5859 - val_loss: 0.5959 - val_accuracy: 0.6993
Epoch 2/6
2311/2311 [==============================] - 928s 402ms/step - loss: 0.6059 - accuracy: 0.6694 - val_loss: 0.5555 - val_accuracy: 0.7150
Epoch 3/6
2311/2311 [==============================] - 928s 401ms/step - loss: 0.5660 - accuracy: 0.7058 - val_loss: 0.5055 - val_accuracy: 0.7641
Epoch 4/6
2311/2311 [==============================] - 928s 402ms/step - loss: 0.5374 - accuracy: 0.7299 - val_loss: 0.4948 - val_accuracy: 0.7576
Epoch 5/6
2311/2311 [==============================] - 929s 402ms/step - loss: 0.5200 - accuracy: 0.7425 - val_loss: 0.4580 - val_accuracy: 0.7907
Epoch 6/6
2311/2311 [==============================] - 927s 401ms/step - loss: 0.5015 - accuracy: 0.7516 - val_loss: 0.4735 - val_accuracy: 0.7641


In [ ]:
history = final_model.fit(
    [X_train_enc['input_ids'], X_train_enc['attention_mask']],
    y_train,
    epochs=6,
    batch_size=16,
    validation_data=([X_test_enc['input_ids'], X_test_enc['attention_mask']], y_test)
)


Epoch 1/6
2022/2022 [==============================] - 936s 450ms/step - loss: 0.6792 - accuracy: 0.5773 - val_loss: 0.6025 - val_accuracy: 0.6975
Epoch 2/6
2022/2022 [==============================] - 917s 453ms/step - loss: 0.6067 - accuracy: 0.6700 - val_loss: 0.5487 - val_accuracy: 0.7319
Epoch 3/6
2022/2022 [==============================] - 928s 459ms/step - loss: 0.5657 - accuracy: 0.7071 - val_loss: 0.5043 - val_accuracy: 0.7658
Epoch 4/6
2022/2022 [==============================] - 948s 469ms/step - loss: 0.5367 - accuracy: 0.7261 - val_loss: 0.4678 - val_accuracy: 0.7884
Epoch 5/6
2022/2022 [==============================] - 923s 456ms/step - loss: 0.5183 - accuracy: 0.7433 - val_loss: 0.4614 - val_accuracy: 0.7861
Epoch 6/6
2022/2022 [==============================] - 935s 462ms/step - loss: 0.5035 - accuracy: 0.7538 - val_loss: 0.4796 - val_accuracy: 0.7599


In [ ]:
history = final_model.fit(
  [X_train_enc['input_ids'], X_train_enc['attention_mask']],
  y_train,
  epochs=6,
  batch_size=18,
  validation_data=([X_test_enc['input_ids'], X_test_enc['attention_mask']], y_test)
)

Epoch 1/6
1797/1797 [==============================] - 828s 443ms/step - loss: 0.6716 - accuracy: 0.5914 - val_loss: 0.6020 - val_accuracy: 0.6486
Epoch 2/6
1797/1797 [==============================] - 802s 446ms/step - loss: 0.6034 - accuracy: 0.6697 - val_loss: 0.5291 - val_accuracy: 0.7601
Epoch 3/6
1797/1797 [==============================] - 801s 446ms/step - loss: 0.5642 - accuracy: 0.7042 - val_loss: 0.4956 - val_accuracy: 0.7751
Epoch 4/6
1797/1797 [==============================] - 802s 446ms/step - loss: 0.5404 - accuracy: 0.7261 - val_loss: 0.5009 - val_accuracy: 0.7459
Epoch 5/6
1797/1797 [==============================] - 802s 446ms/step - loss: 0.5207 - accuracy: 0.7377 - val_loss: 0.4675 - val_accuracy: 0.7777
Epoch 6/6
1797/1797 [==============================] - 801s 446ms/step - loss: 0.5110 - accuracy: 0.7444 - val_loss: 0.4425 - val_accuracy: 0.7988


In [ ]:
history = final_model.fit(
  [X_train_enc['input_ids'], X_train_enc['attention_mask']],
  y_train,
  epochs=6,
  batch_size=20,
  validation_data=([X_test_enc['input_ids'], X_test_enc['attention_mask']], y_test)
)

Epoch 1/6
1618/1618 [==============================] - 820s 498ms/step - loss: 0.6743 - accuracy: 0.5862 - val_loss: 0.5938 - val_accuracy: 0.7543
Epoch 2/6
1618/1618 [==============================] - 809s 500ms/step - loss: 0.6153 - accuracy: 0.6595 - val_loss: 0.5643 - val_accuracy: 0.7036
Epoch 3/6
1618/1618 [==============================] - 803s 496ms/step - loss: 0.5792 - accuracy: 0.6923 - val_loss: 0.5282 - val_accuracy: 0.7330
Epoch 4/6
1618/1618 [==============================] - 802s 496ms/step - loss: 0.5503 - accuracy: 0.7176 - val_loss: 0.4836 - val_accuracy: 0.7754
Epoch 5/6
1618/1618 [==============================] - 802s 496ms/step - loss: 0.5309 - accuracy: 0.7322 - val_loss: 0.4681 - val_accuracy: 0.7790
Epoch 6/6
1618/1618 [==============================] - 801s 495ms/step - loss: 0.5165 - accuracy: 0.7418 - val_loss: 0.4667 - val_accuracy: 0.7767


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
final_model.save('models/fourth_training')

In [ ]:
!cp -r models/third_training drive/MyDrive/third_training

-----
## Ładowanie modelu

In [ ]:
#zaladowanie modelu
loaded_model = tf.keras.models.load_model('drive/MyDrive/third_training')
# Aby uzytkownik mogl zaladowac model uzyty w pracy
# nalezy pobrac folder https://drive.google.com/drive/folders/1dTyamA2eWJkztQZnCH9OlFkDTHFTYflK
#
# nastepnie pobrany folder dac jako argument w ponizszym poleceniu
# loaded_model = tf.keras.models.load_model('tutaj sciezka do folderu')

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def encode_user_text(text):
    return tokenizer(text, padding='max_length', truncation=True, max_length=256, return_tensors='tf')

## Interfejs tekstowy na potrzeby testów

In [ ]:
user_input = 'Great product, size exactly as specified and color is just like in the pictures'
processed_input = encode_user_text(user_input)

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


In [ ]:
prediction = loaded_model.predict([processed_input['input_ids'], processed_input['attention_mask']])
if prediction[0] > 0.5:
  print('This review is written by a human. ')
else:
  print('This review is machine generated. ')
print('prediction = ', prediction[0])

1/1 [==============================] - 0s 211ms/step
This review is written by a human. 
prediction =  [0.6409352]


# Testowanie wygenerowanych opinii

In [ ]:
# funkcja do rozpoznawania kolejno opinii z listy
def recognize_machine_generated(reviews: list):
  machine_generated_amount = 0

  for i in reviews:
    user_input = i
    processed_input = encode_user_text(user_input)

    prediction = loaded_model.predict([processed_input['input_ids'], processed_input['attention_mask']])
    if prediction[0] > 0.5:
      print('This review is written by a human. ')
    else:
      print('This review is machine generated. ')
      machine_generated_amount += 1
    print('prediction = ', prediction[0])

  print(f"Machine generated according to the model: {machine_generated_amount} out of {len(generated_reviews)}")

In [ ]:
# wygenerowane opinie koca (blanket)
generated_reviews = ["Absolutely love this blanket! It's so soft and keeps me warm all night.",
                     "The material feels luxurious, and it holds up well after washing. Highly recommend!",
                     "Lightweight and soft—ideal for curling up on the couch.",
                     "Exactly what I was looking for. Soft, warm, and the color is beautiful.",
                     "Bought one for my friend, and they loved it. Might grab one for myself too.",
                     "The texture feels rougher than it looked in the pictures. Disappointed.",
                     "This blanket sheds everywhere! My clothes and furniture are covered in fuzz.",
                     "Nice design, but it's way too thin to keep you warm in colder weather.",
                     "Came with a strong chemical odor that didn’t go away even after washing.",
                     "The seams started coming apart after just a few uses. Very poor quality."]

print(len(generated_reviews))
recognize_machine_generated(generated_reviews)
print('\n\n')

# opinie koca napisane przez człowieka
generated_reviews = ["Literally the best blanket Ive ever owned. Its really soft and cozy",
                     "Bought it for my wife for xmas and she seems very happy with it. Also it's a pretty good deal",
                     "Its so soft like whaaaaat?? I literally wrap myself in it every night and it's so soo comfy! <333",
                     "I have 2 small children and there obsessed with this blanket! So far it survived being puked on (multiple times), used as a roof of a castle, being thrown out of a window from 2nd floor etc... Suffice to say it's worth its price.",
                     "Good blanket, five stars.",
                     "One star, everyone be careful it melts easily if you drop a cigarette on it and i almos had to rush to ER bc of it :((((",
                     "If I could I would rate this garbage zero stars!!!! First of all, my package came all torn up like someone has just played the whole world cup with it?! Customer service is a joke, I couldn't get a hold of the seller for like solid 2 weeks and when he finally answered he was so rude and arrogant like I'm the one to blame??? Terrible shopping experience, don't buy ANYTHING from him!!!",
                     "It's 100% plastic and smells like chemicals, also it's so fricking small and not as soft as advertised! A waste of money in my opinion.",
                     "My dog ate it, the vet was 400$.",
                     "Poor quality, it's plastic not wool. It's already ripping even though I barely touched it."]
recognize_machine_generated(generated_reviews)

10
1/1 [==============================] - 0s 75ms/step
This review is machine generated. 
prediction =  [0.1359695]
1/1 [==============================] - 0s 82ms/step
This review is written by a human. 
prediction =  [0.59190154]
1/1 [==============================] - 0s 78ms/step
This review is machine generated. 
prediction =  [0.37194797]
1/1 [==============================] - 0s 76ms/step
This review is machine generated. 
prediction =  [0.14008546]
1/1 [==============================] - 0s 79ms/step
This review is written by a human. 
prediction =  [0.6059393]
1/1 [==============================] - 0s 76ms/step
This review is machine generated. 
prediction =  [0.27539164]
1/1 [==============================] - 0s 74ms/step
This review is machine generated. 
prediction =  [0.38178614]
1/1 [==============================] - 0s 76ms/step
This review is machine generated. 
prediction =  [0.26754946]
1/1 [==============================] - 0s 73ms/step
This review is written by a human

In [ ]:
# wygenerowane opinie krzesla biurowego (office chair)
generated_reviews = ["Comfortable for extended use, armrests could offer additional padding.",
                     "Assembly process is straightforward, and lumbar support is effective.",
                     "Design is ergonomic and visually appealing, seat cushion firmness could be improved.",
                     "Adequate for the price, recline function requires increased smoothness.",
                     "Back support is reliable, though armrest height adjustment is limited.",
                     "Build quality is satisfactory; seat padding could benefit from increased density.",
                     "Adjustment options are useful, but chair height range is slightly restricted.",
                     "Provides support and comfort; fabric material is prone to staining.",
                     "Highly suitable for home office environments; adjustment controls are responsive.",
                     "Sturdy and supportive construction; wheel movement could be smoother."]

print(len(generated_reviews))
recognize_machine_generated(generated_reviews)
print('\n\n')

# opinie krzesla biurowego napisane przez człowieka
generated_reviews = ["I bought it for my home office and even though I'm a single mom with three beautiful children I've assembled it with no problems. It is more on the pricey side but I'm really happy with it.",
                     "Sturdy and comfortable, price is reasonable. Hopefully it will last me at least a few years",
                     "I bought it for my son, he was nagging me for a gaming chair but those cost a good fortune so we settled for this one and he seems reallt happy.",
                     "Pretty good chair. I can edit my videos for at least a few hours before I start to feel my back. Arm rests and height are adjustable, headrest is reasonable. Worth every cent.",
                     "Good choice for a small office. I've heard no complaints so far so it looks like the adjustment range is wide enough. The design is so universal it fits into any office space without sticking out like a sore thumb.",
                     "Both of my children got these chairs as they started middle school and even after there growth spurts they still fit them. Highly recommend.",
                     "Nice chair, enjoyed",
                     "It is surprisingly light, which helped me a lot with moving in to my new dorm room. It's comfortable, no complaints there, but there could be more color options",
                     "I sit in a chair for a living haha. As a streamer I need my butt to be as comfortable as it can possibly be and this chair does it's job. It survived my severe gamer rage, drywall: 0 chair: 1",
                     "Quick delivery, product as advertised, no complaints"]
recognize_machine_generated(generated_reviews)

10
1/1 [==============================] - 0s 129ms/step
This review is machine generated. 
prediction =  [0.46216682]
1/1 [==============================] - 0s 159ms/step
This review is machine generated. 
prediction =  [0.4636506]
1/1 [==============================] - 0s 75ms/step
This review is machine generated. 
prediction =  [0.43305966]
1/1 [==============================] - 0s 107ms/step
This review is machine generated. 
prediction =  [0.45459387]
1/1 [==============================] - 0s 106ms/step
This review is machine generated. 
prediction =  [0.32228637]
1/1 [==============================] - 0s 115ms/step
This review is machine generated. 
prediction =  [0.38655064]
1/1 [==============================] - 0s 96ms/step
This review is machine generated. 
prediction =  [0.17416988]
1/1 [==============================] - 0s 92ms/step
This review is machine generated. 
prediction =  [0.34811866]
1/1 [==============================] - 0s 98ms/step
This review is written by a h

In [13]:
# wygenerowane opinie butow (shoes)
generated_reviews = ["These are the most comfortable shoes I’ve ever worn! Perfect for long walks.",
                     "True to size and fits like a glove. No breaking-in needed!",
                     "Love the design, and they’re great for both casual outings and work.",
                     "I’ve been wearing these every day for months, and they still look brand new!",
                     "Affordable, stylish, and comfortable. Can’t ask for more at this price!",
                     "These shoes are way too stiff and gave me blisters after one wear.",
                     "The sole started separating after just two weeks of use. Poor quality.",
                     "Runs way too small. I had to return them and size up.",
                     "They look great, but there’s no arch support, making them uncomfortable for extended wear.",
                     "The material feels cheap, and they started falling apart almost immediately. Not worth it."]
print(len(generated_reviews))
recognize_machine_generated(generated_reviews)
print('\n\n')

# opinie butow napisane przez człowieka
generated_reviews = ["Quick delivery, good quality and sizing. 5 stars",
                     "Great shoes, I really like them and they fit me really well. Wide enough for my chud feet, would recommend",
                     "I bought them as a gift for my wife since she was eyeing them for a while. She wore them to a New Year's Eve party, and even after dancing for the whole night, she didn't feel the need to change them.",
                     "At first I was skeptical since they are quite cheap, but this is indeed real leather. Great fit and quality, probably the best choice in this price range.",
                     "I'm literally obsessed, probably the most eye-catching pair in my collection. Maybe not the most comfortable but for a night out without much walking they are a fantastic choice",
                     "Gorgeous shoes, but incredibly uncomfortable to the point of bleeding after just an hour of wearing them. I don't think they would fit any living person",
                     "They don't look anything like on the photos. Must be that AI or something. Anyways don't buy them.",
                     "From afar they look okay but when you get closer they look very cheap and poorly made",
                     "I have no idea what happened but they came all scratched up and dirty like someone's been wearing them already?? I've asked for a refund and got it but still very weird",
                     "I've made a mistake of wearing white socks with them, and let's just say I no longer have white socks. Generally very poor quality"]
recognize_machine_generated(generated_reviews)

10
1/1 [==============================] - 0s 130ms/step
This review is machine generated. 
prediction =  [0.36730504]
1/1 [==============================] - 0s 177ms/step
This review is written by a human. 
prediction =  [0.87368983]
1/1 [==============================] - 0s 112ms/step
This review is machine generated. 
prediction =  [0.45063484]
1/1 [==============================] - 0s 128ms/step
This review is machine generated. 
prediction =  [0.30117172]
1/1 [==============================] - 0s 113ms/step
This review is written by a human. 
prediction =  [0.82525086]
1/1 [==============================] - 0s 129ms/step
This review is machine generated. 
prediction =  [0.32514188]
1/1 [==============================] - 0s 131ms/step
This review is machine generated. 
prediction =  [0.3088672]
1/1 [==============================] - 0s 140ms/step
This review is written by a human. 
prediction =  [0.64875245]
1/1 [==============================] - 0s 121ms/step
This review is machine